<div align="center">

# Week 14: Dependability, Safety and Verification

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-14) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/Week14_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines `lab_rng`, the random generator of the interactive lab, the probabilities of the island supply and the battery schedule of Week 9. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
from collections import deque
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

def lab_rng(seed):
    """The random generator of the interactive lab, so that the Colab notebook draws exactly the same numbers."""
    s = seed & 0xFFFFFFFF
    def uniform():
        nonlocal s
        s = (s + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((s ^ (s >> 15)) * (s | 1)) & 0xFFFFFFFF
        t = (t ^ ((t + (((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF)) & 0xFFFFFFFF)) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    return uniform

# The island supply of the critical loads: the server room of the library B2 and the freezers of the laboratories B5
F_GRID = 2.0                      # outages of the grid per year that need the island
LOW_HOURS = (20, 21, 22, 23)      # hours that start with the battery below the reserve in the schedules of Week 9
Q_BMS = 0.004                     # the battery management system blocks the discharge (35 hours a year)
Q_INV = 0.0082                    # the inverter is under repair (MTTF 43800 h, MTTR 360 h)
Q_BRK = 0.003                     # the breaker at the point of common coupling fails to open
LAM_C, BETA = 0.1, 0.1            # hidden faults of an island controller per year, share of common causes
COST_RESERVE, COST_CTRL, COST_TEST = 3000, 600, 250   # euro a year, euro a year, euro per proof test
TARGET = 0.05                     # tolerable losses of the critical supply per year

# The battery schedule of Week 9: the net demand of a clear and a cloudy day (kW) and the price of each hour (euro/kWh)
NET_CLEAR = np.array([196.2, 183.7, 172.9, 176.7, 191.5, 223.6, 250.4, 294.6, 359.0, 352.4, 304.5, 259.6,
                      233.7, 246.9, 281.7, 341.7, 411.8, 502.9, 580.5, 585.7, 489.9, 401.7, 295.3, 233.6])
NET_CLOUDY = np.array([196.2, 183.7, 172.9, 176.7, 191.5, 223.6, 279.4, 376.2, 484.5, 511.6, 486.9, 455.0,
                       432.6, 440.2, 460.1, 496.1, 532.8, 581.4, 608.3, 585.7, 489.9, 401.7, 295.3, 233.6])
PRICE = np.array([0.14, 0.1393, 0.1389, 0.1388, 0.1389, 0.1396, 0.1418, 0.146, 0.1525, 0.156, 0.1575, 0.158,
                  0.157, 0.1565, 0.156, 0.1555, 0.157, 0.17, 0.22, 0.24, 0.21, 0.1575, 0.1475, 0.1422])
E_KWH, E_MIN, E_MAX, E0, P_BAT, ETA, IMPORT_MAX = 500.0, 50.0, 450.0, 250.0, 250.0, 0.95, 600.0
S_EXPORT, K_PEAK, V_STORED = 0.05, 0.3, 0.15
RESERVE_KWH = 130.0               # the floor of 50 kWh plus one hour of the critical loads, 80 kW
print("Setup complete. Running in Colab:", IN_COLAB, "| outages per year:", F_GRID, "| reserve:", RESERVE_KWH, "kWh")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 14.1": (lambda: 0.95 * 0.98 * 0.99, 1e-9),
    "Exercise 14.2": (lambda: 1 - 0.1 ** 3, 1e-9),
    "Exercise 14.3": (lambda: 28.5, 1e-9),
    "Exercise A": (lambda: len(model_check(2)['states']), 1e-9),
    "Exercise B": (lambda: k_out_of_n(reliability(0.1, 5.0), 2, 3), 1e-9),
    "Exercise C": (lambda: F_GRID * p_loss(events(True, 2, 6)), 1e-9),
    "Exercise D": (lambda: len(monitor(schedule(NET_CLOUDY)['bat_kw'])[2]), 1e-9),
    "Exercise E": (lambda: 1 / STAGE14['critical supply, chosen design: losses per year'], 1e-6),
}

## Python step 14: Organised and tested code

Safety-related code must be easy to find, to reuse and to check. This step puts functions into a module of their own, states assumptions with `assert` and runs small tests, ending with a monitor that guards the reserve of the battery &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "NET_CLEAR" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-14/"
               "NB14_dependability_safety_verification.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Functions in a module

In [ ]:
# Python practice: a small example for learning the language
module_code = """# campus_safety.py: small tools for the safety of the campus microgrid

def series(parts):
    # parts that must all work: the reliabilities multiply
    r = 1.0
    for p in parts:
        r *= p
    return r

def parallel(parts):
    # parts of which one is enough: the unreliabilities multiply
    q = 1.0
    for p in parts:
        q *= 1 - p
    return 1 - q
"""
with open("campus_safety.py", "w") as f:
    f.write(module_code)

import importlib
import campus_safety
importlib.reload(campus_safety)          # reads the file again if it has changed
print("in series:", round(campus_safety.series([0.99, 0.95]), 4))
print("in parallel:", round(campus_safety.parallel([0.9, 0.9]), 4))

A module is a file of Python code. The statement `import campus_safety` runs the file once and makes its functions available as `campus_safety.series` and `campus_safety.parallel`. The function `reload` reads the file again after a change. Larger projects keep their models, monitors and tests in such modules.

### Assertions

In [ ]:
# Python practice: a small example for learning the language
r = campus_safety.parallel([0.9, 0.9])
assert 0.0 <= r <= 1.0, "a reliability lies between 0 and 1"
assert r > 0.9, "two parts in parallel are better than one"
print("both assertions hold for", round(r, 4))
try:
    assert campus_safety.series([0.9, 0.9]) > 0.9, "two parts in series are better than one"
except AssertionError as err:
    print("assertion failed:", err)

The statement `assert` checks a condition and stops the program with an `AssertionError` if it is false. The text after the comma explains what was expected. The block `try` and `except` catches the error here, so the cell can show the message and continue.

### Tests and a small safety monitor

In [ ]:
# Python practice: a small example for learning the language
def allowed_discharge(e_kwh, p_kw, reserve_kwh=130.0, eta=0.95):
    # the largest discharge in one hour that keeps the reserve in the battery
    return max(0.0, min(p_kw, (e_kwh - reserve_kwh) * eta))

def test_keeps_reserve():
    e = 150.0
    assert e - allowed_discharge(e, 100.0) / 0.95 >= 130.0 - 1e-9

def test_full_battery_unchanged():
    assert allowed_discharge(450.0, 100.0) == 100.0

def test_never_negative():
    assert allowed_discharge(120.0, 50.0) == 0.0

TESTS = [test_keeps_reserve, test_full_battery_unchanged, test_never_negative]
passed = 0
for test in TESTS:
    try:
        test()
        passed += 1
        print("passed:", test.__name__)
    except AssertionError:
        print("FAILED:", test.__name__)
print(passed, "of", len(TESTS), "tests passed")

A test is a small function that calls the code with chosen inputs and asserts the expected result. The list `TESTS` holds the functions themselves, and the loop calls each one and counts the passes. Tools such as pytest find and run such functions automatically.

**Quick check.** Why does a failed assertion help more than a program that simply prints a wrong number?

<details><summary>Answer</summary>

The assertion stops the program at the place where an assumption breaks and says which one. A wrong number may travel unnoticed into a schedule, a report or the controller of the battery.

</details>

### Exercises for Python step 14

**Exercise 14.1.** Store the reliability of three parts in series with 0.95, 0.98 and 0.99, computed with `campus_safety.series`, in `r_141`.

In [ ]:
r_141 = None   # your answer

check("Exercise 14.1", r_141)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r_141 = campus_safety.series([0.95, 0.98, 0.99])

check("Exercise 14.1", r_141)

**Exercise 14.2.** Store the reliability of three controllers in parallel, each with 0.9, computed with `campus_safety.parallel`, in `r_142`.

In [ ]:
r_142 = None   # your answer

check("Exercise 14.2", r_142)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r_142 = campus_safety.parallel([0.9, 0.9, 0.9])

check("Exercise 14.2", r_142)

**Exercise 14.3.** A battery holds 160 kWh and is asked for 50 kW. Store the discharge in kW that `allowed_discharge` permits in `p_143`.

In [ ]:
p_143 = None   # your answer

check("Exercise 14.3", p_143)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_143 = allowed_discharge(160.0, 50.0)

check("Exercise 14.3", p_143)

## 1. Model checking: The transfer logic of the island controller &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#ref-3)&#93;

*Lecture: [Verification: Checking every state](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#model-checking)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "allowed_discharge" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-14/"
               "NB14_dependability_safety_verification.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** When the grid fails, the island controller opens the breaker at the point of common coupling and lets the inverter form the island grid. If the inverter formed the grid with the breaker still closed, the battery would energise the lines of the grid during a repair. Python builds a model of the transfer logic and visits every state that it can reach.

**From the formulas to the code.** A state is the tuple (mode, timer, breaker, inverter, grid, stuck), and one step lasts 100 ms: `step`. The safety invariant forbids $\text{inverter forms the grid} \wedge \text{breaker closed}$ in every reachable state: `unsafe`. Breadth-first search, `model_check(version)`, visits each state once and keeps the state it came from, so the shortest path to an unsafe state can be traced back.

In [ ]:
INIT = ("GRID", 0, 1, "follow", 1, 0)    # mode, timer, breaker closed, inverter follows, grid ok, not stuck

def step(state, grid, stuck, version):
    """One step of 100 ms: the grid and a stuck breaker are chosen first, then the controller reacts and the breaker
    moves. Version 1 trusts a timer, version 2 waits for the breaker to report open."""
    mode, t, brk, inv = state[:4]
    cmd = None
    if mode == "GRID":
        inv = "follow"
        if not grid:
            mode, t, inv, cmd = "OPENING", 0, "off", "open"
    elif mode == "OPENING":
        cmd = "open"
        if version == 1:
            if t >= 1:
                mode, t, inv = "ISLAND", 0, "form"         # 200 ms after the command: assumes an open breaker
            else:
                t += 1
        elif brk == 0:
            mode, t, inv = "ISLAND", 0, "form"             # the breaker reports open
        elif t >= 2:
            mode, t, inv = "FAULT", 0, "off"               # still closed after 300 ms: stop safely
        else:
            t += 1
    elif mode == "ISLAND":
        if grid:
            mode, t = "SYNC", 0
    elif mode == "SYNC":
        if not grid:
            mode, t = "ISLAND", 0
        elif t >= 1:
            mode, t, inv, cmd = "GRID", 0, "follow", "close"
        else:
            t += 1
    elif mode == "FAULT":
        if grid and brk == 1:
            mode, inv = "GRID", "follow"
    if cmd == "open" and not stuck:
        brk = 0
    elif cmd == "close":
        brk = 1
    return (mode, t, brk, inv, grid, stuck)

def unsafe(state):
    return state[3] == "form" and state[2] == 1     # the inverter forms the grid with the breaker closed

def model_check(version):
    """Breadth-first search over all reachable states. Returns the states with their parents, the unsafe states and
    the shortest path to an unsafe state."""
    parent, depth, queue, bad = {INIT: None}, {INIT: 0}, deque([INIT]), []
    while queue:
        s = queue.popleft()
        if unsafe(s):
            bad.append(s)
        for grid in (0, 1):
            for stuck in ((1,) if s[5] else (0, 1)):     # a stuck breaker stays stuck
                n = step(s, grid, stuck, version)
                if n not in parent:
                    parent[n], depth[n] = s, depth[s] + 1
                    queue.append(n)
    path = []
    if bad:
        x = min(bad, key=lambda b: depth[b])
        while x is not None:
            path.append(x)
            x = parent[x]
        path.reverse()
    return dict(states=parent, depth=depth, unsafe=bad, path=path)

RESULT = {v: model_check(v) for v in (1, 2)}
for v, res in RESULT.items():
    print(f"version {v}: {len(res['states'])} reachable states, {len(res['unsafe'])} unsafe, "
          f"{max(res['depth'].values()) + 1} layers")
print("shortest counterexample of version 1:")
for k, s in enumerate(RESULT[1]["path"]):
    print(f"  step {k}: mode {s[0]:<8} timer {s[1]}  breaker {'closed' if s[2] else 'open':<6}  inverter {s[3]:<6}  "
          f"grid {'ok' if s[4] else 'lost'}  stuck {s[5]}")

SCENARIOS = {  # grid and stuck breaker in the steps 1 to 12 of the test bench
    "none": ([0] * 6 + [1] * 6, [0] * 12),
    "stuck": ([0] * 6 + [1] * 6, [1] * 12),
    "dip": ([0] + [1] * 11, [0] * 12),
}

def run_scenario(version, fault):
    """One test on the bench: returns the trace, the step of a hazard and the outcome."""
    grid, stuck = SCENARIOS[fault]
    s, trace, hazard = INIT, [INIT], None
    for k in range(12):
        s = step(s, grid[k], stuck[k], version)
        trace.append(s)
        if hazard is None and unsafe(s):
            hazard = k + 1
    outcome = ("back-feed" if hazard else "safe stop" if any(x[0] == "FAULT" for x in trace) else "reconnected")
    return dict(trace=trace, hazard=hazard, outcome=outcome)

for v in (1, 2):
    for fault in SCENARIOS:
        r = run_scenario(v, fault)
        print(f"test: version {v}, fault {fault:<5} -> {r['outcome']}" + (f" in step {r['hazard']}" if r["hazard"] else ""))

**Python note.** The class `deque` is a queue with fast access at both ends, and `popleft` takes the oldest state first, which makes the search breadth-first. Tuples can be keys of a dictionary, so `parent` maps every state to the state it came from.

**Exercise A.** Store the number of reachable states of the corrected logic, version 2, in `n_a`.

In [ ]:
n_a = None   # your answer

check("Exercise A", n_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_a = len(model_check(2)['states'])

check("Exercise A", n_a)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lab.html) opens the transfer test bench. Paste the copied rows into `MY_TESTS`, and the code repeats each of your tests.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_TESTS = """
"""
if MY_TESTS.strip():
    tests = pd.read_csv(io.StringIO(MY_TESTS.strip()))
    source = "your own tests on the bench"
else:
    tests = pd.DataFrame({"run": [1, 2, 3], "logic": [1, 1, 2], "fault": ["none", "stuck", "stuck"],
                          "outcome": ["reconnected", "back-feed", "safe stop"]})
    source = "a simulated student"
tests["model_outcome"] = [run_scenario(int(r.logic), r.fault)["outcome"] for r in tests.itertuples()]
print("Data:", source)
print(tests.to_string(index=False))

In [ ]:
# Figure: the reachable states of the transfer logic in both versions
COLORS = {"GRID": "#1E6FA8", "OPENING": "#C98510", "ISLAND": "#2E8B57", "SYNC": "#0E7A78", "FAULT": "#56616D"}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for a, v in zip(ax, (1, 2)):
    res = RESULT[v]
    layers = {}
    for s, d in res["depth"].items():
        layers.setdefault(d, []).append(s)
    pos = {s: (d, (i - (len(ss) - 1) / 2)) for d, ss in layers.items() for i, s in enumerate(ss)}
    for s, p in res["states"].items():
        if p is not None:
            a.plot([pos[p][0], pos[s][0]], [pos[p][1], pos[s][1]], color="#9DB7CC", lw=0.8, zorder=1)
    on_path = set(res["path"])
    for s, (x, y) in pos.items():
        a.scatter(x, y, s=150 if s in on_path else 90, color="#B53F2E" if unsafe(s) else COLORS[s[0]],
                  edgecolor="#1B2430" if s in on_path else "white", lw=1.5, zorder=2)
    a.set_title(f"version {v}: {len(res['states'])} states, {len(res['unsafe'])} unsafe", fontsize=9)
    a.set_xlabel("steps of 100 ms from the start")
for mode, col in COLORS.items():
    ax[1].scatter([], [], color=col, label=mode)
ax[1].scatter([], [], color="#B53F2E", label="unsafe")
ax[1].legend(fontsize=7, loc="upper right")
ax[0].set_yticks([])
plt.tight_layout()
plt.show()

**Observe and reflect.** Every test of the first version without a stuck breaker passed. What does that say about the tests that a team would have written, compared with the model checker?

## 2. Reliability and redundancy &#91;[6](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#ref-6)&#93;

*Lecture: [Reliability and redundancy](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#reliability)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "run_scenario" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-14/"
               "NB14_dependability_safety_verification.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Parts of the island supply fail in operation, and redundancy tolerates some of these failures. Python computes the reliability of parts in series and in parallel, of two-out-of-three voting and the availability of a part that is repaired after a failure.

**From the formulas to the code.** A constant failure rate $\lambda$ gives $R(t) = e^{-\lambda t}$ and the mean time to failure (MTTF) $1/\lambda$: `reliability`. In series $R_s = \prod R_i$: `series`; in parallel $R_p = 1 - \prod (1 - R_i)$: `parallel`. A k-out-of-n system works with probability $\sum_{j=k}^{n} \binom{n}{j} R^j (1 - R)^{n - j}$: `k_out_of_n`. With the mean time to repair (MTTR), a repaired part is available with $A = \text{MTTF}/(\text{MTTF} + \text{MTTR})$: `availability`.

In [ ]:
def reliability(lam, t):
    return math.exp(-lam * t)

def series(parts):
    return float(np.prod(parts))

def parallel(parts):
    return 1 - float(np.prod([1 - p for p in parts]))

def k_out_of_n(r, k, n):
    return sum(math.comb(n, j) * r ** j * (1 - r) ** (n - j) for j in range(k, n + 1))

def availability(mttf_h, mttr_h):
    return mttf_h / (mttf_h + mttr_h)

LAMBDAS = {"battery management": 0.05, "inverter": 0.2, "island controller": 0.1, "breaker": 0.02}   # per year
lam_s = sum(LAMBDAS.values())
print(f"island supply in series: failure rate {lam_s:.2f} per year, R(1 year) = "
      f"{series([reliability(l, 1.0) for l in LAMBDAS.values()]):.4f}, MTTF {1 / lam_s:.2f} years")
r1 = reliability(0.1, 1.0)
print(f"one controller after a year: {r1:.4f} | two in parallel: {parallel([r1, r1]):.4f} | "
      f"two out of three: {k_out_of_n(r1, 2, 3):.4f}")
print("MTTF in years: one 10.00 | one out of two", round(3 / (2 * 0.1), 2), "| two out of three", round(5 / (6 * 0.1), 2))
print(f"two out of three is better than one controller until {math.log(2) / 0.1:.2f} years")
a_inv = availability(43800, 360)
print(f"inverter: availability {a_inv:.5f}, unavailability {1 - a_inv:.5f}")

**Python note.** The function `math.comb(n, j)` counts the ways to choose j of n parts. A generator expression inside `sum` adds the terms without building a list first.

**Exercise B.** Store the reliability of two out of three island controllers after 5 years, with a failure rate of 0.1 per year, in `r_b`.

In [ ]:
r_b = None   # your answer

check("Exercise B", r_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r_b = k_out_of_n(reliability(0.1, 5.0), 2, 3)

check("Exercise B", r_b)

In [ ]:
t = np.linspace(0, 20, 201)
r = np.exp(-0.1 * t)
plt.figure(figsize=(8, 3.4))
plt.plot(t, r, color="#56616D", label="one controller")
plt.plot(t, 1 - (1 - r) ** 2, color="#1E6FA8", label="two in parallel (one out of two)")
plt.plot(t, 3 * r ** 2 - 2 * r ** 3, color="#C98510", label="two out of three")
plt.axvline(math.log(2) / 0.1, color="#B53F2E", ls=":", lw=1)
plt.xlabel("years")
plt.ylabel("reliability")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** Why does two-out-of-three voting fall below a single controller after about seven years, and what does that mean for the maintenance of redundant parts?

## 3. The fault tree of the critical supply &#91;[8](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#ref-8)&#93;

*Lecture: [Fault trees](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#fault-trees)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "k_out_of_n" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-14/"
               "NB14_dependability_safety_verification.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** In an outage of the grid, the critical loads lose their supply if any of five basic events occurs. The battery may be below its reserve, the battery management system may block the discharge or the inverter may be under repair. Both island controllers may have failed, or the breaker may fail to open. Python builds this fault tree, compares twelve designs and checks the tree with 50,000 simulated years.

**From the formulas to the code.** Hidden faults with proof tests every $\tau$ years give $q_1 = \lambda\tau/2$ for one controller and $q_{1oo2} = \beta\lambda\tau/2 + ((1 - \beta)\lambda\tau)^2/3$ for two: `q_controllers`. The OR gate over independent events gives $P_{loss} = 1 - \prod (1 - q_i)$: `p_loss`, and the losses per year are $f = f_{grid} P_{loss}$. The Fussell-Vesely importance is $q_i \prod_{j \ne i}(1 - q_j)/P_{loss}$: `importance`.

In [ ]:
NAMES = {"R": "battery below the reserve", "B": "battery management blocks", "I": "inverter under repair",
         "C": "island controllers fail", "K": "breaker fails to open"}

def q_controllers(n, months):
    tau = months / 12
    if n == 1:
        return LAM_C * tau / 2
    return BETA * LAM_C * tau / 2 + ((1 - BETA) * LAM_C * tau) ** 2 / 3

def events(reserve, n, months):
    """The probabilities of the basic events in an outage, for a design."""
    return dict(R=0.0 if reserve else len(LOW_HOURS) / 24, B=Q_BMS, I=Q_INV, C=q_controllers(n, months), K=Q_BRK)

def p_loss(q):
    survive = 1.0
    for v in q.values():
        survive *= 1 - v
    return 1 - survive

def importance(q):
    total = p_loss(q)
    return {k: v * (1 - total) / (1 - v) / total for k, v in q.items()}

def cost(reserve, n, months):
    return (COST_RESERVE if reserve else 0) + (COST_CTRL if n == 2 else 0) + n * (12 // months) * COST_TEST

rows = []
for reserve in (False, True):
    for n in (1, 2):
        for months in (12, 6, 3):
            q = events(reserve, n, months)
            rows.append(dict(reserve=reserve, controllers=n, test_months=months, q_controllers=round(q["C"], 6),
                             p_loss=round(p_loss(q), 5), losses_per_year=round(F_GRID * p_loss(q), 4),
                             cost_eur=cost(reserve, n, months)))
DESIGNS = pd.DataFrame(rows)
DESIGNS["meets_target"] = DESIGNS.losses_per_year < TARGET
print(DESIGNS.to_string(index=False))
ok = DESIGNS[DESIGNS.meets_target]
CHOSEN = ok.loc[ok.cost_eur.idxmin()]
print("cheapest design below the target:", dict(CHOSEN))
q_now = events(False, 2, 12)
print("current design: P_loss", round(p_loss(q_now), 4), "| losses per year", round(F_GRID * p_loss(q_now), 4),
      "| one loss in", round(1 / (F_GRID * p_loss(q_now)), 2), "years")
for k, v in importance(q_now).items():
    print(f"  importance of {NAMES[k]:<28} {v:.4f}")

**Python note.** The dictionary `NAMES` turns the short keys of the basic events into words for the output. The method `idxmin` returns the row with the smallest cost, and `.loc` picks that row.

**Exercise C.** Store the losses per year of the design with the reserve, two controllers and proof tests every 6 months in `f_c`.

In [ ]:
f_c = None   # your answer

check("Exercise C", f_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
f_c = F_GRID * p_loss(events(True, 2, 6))

check("Exercise C", f_c)

In [ ]:
def simulate(reserve, n, months, years=50000, seed=1405):
    """Outages year by year with the random numbers of the interactive lab: the hour of each outage, the basic events
    and the time since the last proof test of the controllers."""
    u = lab_rng(seed)
    tau, limit = months / 12, math.exp(-F_GRID)
    losses = outages = 0
    for _ in range(years):
        k, p = 0, u()
        while p > limit:                                # the number of outages in the year: Poisson with mean 2
            k += 1
            p *= u()
        for _ in range(k):
            outages += 1
            hour = int(u() * 24)
            b, i, s, ccf, a1, a2, kb = u(), u(), u() * tau, u(), u(), u(), u()
            low = (not reserve) and hour in LOW_HOURS
            if n == 1:
                ctrl = ccf < BETA * LAM_C * s or a1 < (1 - BETA) * LAM_C * s
            else:
                ctrl = ccf < BETA * LAM_C * s or (a1 < (1 - BETA) * LAM_C * s and a2 < (1 - BETA) * LAM_C * s)
            if low or b < Q_BMS or i < Q_INV or ctrl or kb < Q_BRK:
                losses += 1
    return dict(losses=losses, outages=outages, losses_per_year=losses / years)

SIM = {}
for design in ((False, 2, 12), (True, 2, 12), (True, 1, 3), (True, 2, 3)):
    SIM[design] = simulate(*design)
    print(f"reserve {design[0]!s:<5} controllers {design[1]} tests every {design[2]:>2} months: "
          f"tree {F_GRID * p_loss(events(*design)):.4f}, simulation {SIM[design]['losses_per_year']:.4f} losses per year "
          f"({SIM[design]['losses']} losses in {SIM[design]['outages']} outages)")

**Python note.** The number of outages in a year is drawn with the method of Knuth: Uniform numbers are multiplied until the product falls below $e^{-2}$. Every outage draws the same eight numbers in every design, so the designs meet exactly the same outages.

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lab.html) opens the electrical room of the campus. Paste the copied rows into `MY_DESIGNS`, and the code repeats each of your designs.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_DESIGNS = """
"""
if MY_DESIGNS.strip():
    mine = pd.read_csv(io.StringIO(MY_DESIGNS.strip()))
    source = "your own designs in the electrical room"
else:
    mine = pd.DataFrame({"run": [1, 2, 3], "reserve": [0, 1, 1], "controllers": [2, 1, 2], "test_months": [12, 3, 12]})
    source = "a simulated student"
mine["model_f_tree"] = [round(F_GRID * p_loss(events(bool(r.reserve), int(r.controllers), int(r.test_months))), 4)
                        for r in mine.itertuples()]
mine["model_f_sim"] = [simulate(bool(r.reserve), int(r.controllers), int(r.test_months))["losses_per_year"]
                       for r in mine.itertuples()]
mine["model_cost_eur"] = [cost(bool(r.reserve), int(r.controllers), int(r.test_months)) for r in mine.itertuples()]
print("Data:", source)
print(mine.to_string(index=False))

In [ ]:
# Figure: the losses per year and the yearly cost of twelve designs
plt.figure(figsize=(8, 3.8))
for r in DESIGNS.itertuples():
    plt.scatter(r.cost_eur, r.losses_per_year, s=60, marker="o" if r.controllers == 2 else "s",
                color="#1E6FA8" if r.reserve else "#C98510", edgecolor="#1B2430" if r.meets_target else "none")
plt.axhline(TARGET, color="#B53F2E", ls=":", lw=1)
plt.annotate("cheapest below the target", (CHOSEN.cost_eur, CHOSEN.losses_per_year), xytext=(4300, 0.12), fontsize=8,
             arrowprops=dict(arrowstyle="-", lw=0.8))
plt.yscale("log")
plt.xlabel("yearly cost (euro)")
plt.ylabel("losses of the critical supply per year")
plt.scatter([], [], color="#C98510", label="no reserve rule")
plt.scatter([], [], color="#1E6FA8", label="reserve rule")
plt.scatter([], [], color="#56616D", marker="s", label="one controller")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** One controller with proof tests every 3 months costs 100 euro less than the chosen design. Why does it still miss the target, and what would have to change for it to meet it?

## 4. A runtime monitor for the reserve &#91;[9](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#ref-9)&#93;

*Lecture: [Runtime monitoring and the simplex architecture](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#runtime-monitoring)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "simulate" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-14/"
               "NB14_dependability_safety_verification.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The reserve rule must hold even when the schedule that drives the battery is wrong. Python writes a monitor that clips every discharge that would take the battery below the reserve and tests it with assertions. It then compares the monitor with a schedule that plans with the reserve from the start.

**From the formulas to the code.** The monitor allows $d_t \le (e_{t-1} - e_{res})\,\eta$ in hour $t$, with $e_{res} = 130$ kWh: `monitor`. The day costs the energy at the hourly prices, minus 0.05 euro/kWh for the export, plus 0.30 euro/kW for the peak of the import, as in Week 9: `day_cost`. The linear program of Week 9 with the lower limit $e_t \ge e_{min}$ is `schedule(net, e_min)`.

In [ ]:
def schedule(net, e_min=E_MIN):
    """The cheapest day of Week 9 as a linear program, with the stored energy kept between e_min and E_MAX."""
    T = 24
    n = 5 * T + 1                                   # charge, discharge, import, export, energy for every hour, peak
    C, D, Gr, X, E, PK = 0, T, 2 * T, 3 * T, 4 * T, 5 * T
    c = np.zeros(n)
    c[Gr:Gr + T], c[X:X + T], c[PK] = PRICE, -S_EXPORT, K_PEAK
    A_eq, b_eq, A_ub, b_ub = [], [], [], []
    for t in range(T):
        row = np.zeros(n)
        row[[Gr + t, X + t, C + t, D + t]] = [1, -1, -1, 1]          # import - export - charge + discharge = net
        A_eq.append(row)
        b_eq.append(net[t])
        row = np.zeros(n)
        row[[E + t, C + t, D + t]] = [1, -ETA, 1 / ETA]               # the energy of the battery
        if t > 0:
            row[E + t - 1] = -1
        A_eq.append(row)
        b_eq.append(E0 if t == 0 else 0.0)
        row = np.zeros(n)
        row[[Gr + t, PK]] = [1, -1]                                     # import below the peak
        A_ub.append(row)
        b_ub.append(0.0)
    row = np.zeros(n)
    row[E + T - 1] = -1                                                 # end the day with at least E0
    A_ub.append(row)
    b_ub.append(-E0)
    bounds = [(0, P_BAT)] * 2 * T + [(0, IMPORT_MAX)] * T + [(0, None)] * T + [(e_min, E_MAX)] * T + [(0, None)]
    res = linprog(c, A_ub=np.array(A_ub), b_ub=b_ub, A_eq=np.array(A_eq), b_eq=b_eq, bounds=bounds, method="highs")
    return dict(cost=res.fun, bat_kw=res.x[D:D + T] - res.x[C:C + T])

def energy(plan_kw, e0=E0):
    e, out = e0, []
    for p in plan_kw:
        e = e + (-p * ETA if p < 0 else -p / ETA)
        out.append(e)
    return np.array(out)

def day_cost(net, plan_kw):
    g = np.asarray(net) - np.asarray(plan_kw)
    return float(PRICE @ np.maximum(g, 0) - S_EXPORT * np.maximum(-g, 0).sum() + K_PEAK * g.max())

def monitor(plan_kw, e0=E0, reserve=RESERVE_KWH):
    """Clips every discharge that would take the battery below the reserve. Returns the safe plan, the energy after
    every hour and the hours in which the monitor acted."""
    e, safe, out, acted = e0, [], [], []
    for t, p in enumerate(plan_kw):
        if p > 0:
            allowed = max(0.0, (e - reserve) * ETA)
            if p > allowed + 1e-6:
                acted.append(t)
                p = allowed
        e = e + (-p * ETA if p < 0 else -p / ETA)
        safe.append(p)
        out.append(e)
    return np.array(safe), np.array(out), acted

DAYS = {}
for name, net in (("clear", NET_CLEAR), ("cloudy", NET_CLOUDY)):
    best = schedule(net)
    safe, e_safe, acted = monitor(best["bat_kw"])
    planned = schedule(net, RESERVE_KWH)
    DAYS[name] = dict(best=best, safe=safe, acted=acted, planned=planned)
    print(f"{name} day: optimal {best['cost']:.2f} euro | with the monitor {day_cost(net, safe):.2f} euro, "
          f"interventions in the hours {acted}, {e_safe[-1]:.1f} kWh at midnight | planned with the reserve "
          f"{planned['cost']:.2f} euro, +{round(planned['cost'], 2) - round(best['cost'], 2):.2f}")
extra = np.mean([round(DAYS[d]["planned"]["cost"], 2) - round(DAYS[d]["best"]["cost"], 2) for d in DAYS])
print(f"the reserve costs about {extra * 365:.0f} euro a year if the year consisted of such days")

**Python note.** The function `monitor` returns three values at once, and `safe, e_safe, acted = monitor(...)` unpacks them. The list `acted` holds the hours of the interventions, so its length counts them.

In [ ]:
def test_monitor_keeps_reserve():
    safe, e, acted = monitor(DAYS["clear"]["best"]["bat_kw"])
    assert e.min() >= RESERVE_KWH - 1e-6, "the stored energy fell below the reserve"

def test_monitor_leaves_safe_plan_alone():
    plan = DAYS["clear"]["planned"]["bat_kw"]
    safe, e, acted = monitor(plan)
    assert acted == [] and np.allclose(safe, plan), "the monitor changed a safe plan"

def test_monitor_never_adds_discharge():
    plan = DAYS["clear"]["best"]["bat_kw"]
    safe, e, acted = monitor(plan)
    assert np.all(safe <= plan + 1e-9), "the monitor discharged more than the plan"

passed = 0
for test in (test_monitor_keeps_reserve, test_monitor_leaves_safe_plan_alone, test_monitor_never_adds_discharge):
    try:
        test()
        passed += 1
        print("passed:", test.__name__)
    except AssertionError as err:
        print("FAILED:", test.__name__, err)
print(passed, "of 3 tests passed")

**Exercise D.** Store the number of hours in which the monitor acts on the optimal schedule of the cloudy day in `n_d`.

In [ ]:
n_d = None   # your answer

check("Exercise D", n_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_d = len(monitor(schedule(NET_CLOUDY)['bat_kw'])[2])

check("Exercise D", n_d)

In [ ]:
hours = np.arange(1, 25)
d = DAYS["clear"]
plt.figure(figsize=(8, 3.4))
plt.plot(hours, energy(d["best"]["bat_kw"]), color="#C98510", marker="o", ms=3, label="optimal schedule")
plt.plot(hours, energy(d["safe"]), color="#B53F2E", ls="--", label="with the monitor")
plt.plot(hours, energy(d["planned"]["bat_kw"]), color="#1E6FA8", lw=2, label="planned with the reserve")
plt.axhline(RESERVE_KWH, color="#56616D", ls=":", lw=1)
plt.text(0.5, RESERVE_KWH + 6, "reserve 130 kWh", fontsize=8)
plt.xlabel("end of the hour")
plt.ylabel("stored energy (kWh)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** The monitor acts late, in the hours of the evening peak, and the re-planned schedule acts early. Why is the second one cheaper, and why should the monitor stay anyway?

## 5. The campus microgrid, stage 14

*Lecture: [The campus microgrid, stage 14: The safety case](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#the-campus-microgrid-stage-14)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "monitor" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-14/"
               "NB14_dependability_safety_verification.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 14 assembles the safety case of the island supply. Its claims rest on the evidence of Sections 1 to 4. They are the model checking of the transfer logic, the fault tree with its simulation and the runtime monitor of the reserve.

In [ ]:
chosen = (bool(CHOSEN.reserve), int(CHOSEN.controllers), int(CHOSEN.test_months))
STAGE14 = {
    "transfer logic, first version: reachable states": len(RESULT[1]["states"]),
    "transfer logic, first version: unsafe states": len(RESULT[1]["unsafe"]),
    "transfer logic, first version: steps to the hazard": len(RESULT[1]["path"]) - 1,
    "transfer logic, corrected: reachable states": len(RESULT[2]["states"]),
    "transfer logic, corrected: unsafe states": len(RESULT[2]["unsafe"]),
    "critical supply, current design: losses per year": round(F_GRID * p_loss(events(False, 2, 12)), 4),
    "critical supply, chosen design: losses per year": round(F_GRID * p_loss(events(*chosen)), 4),
    "critical supply, chosen design: yearly cost (euro)": int(CHOSEN.cost_eur),
    "critical supply, chosen design: simulated losses per year": simulate(*chosen)["losses_per_year"],
    "reserve: interventions of the monitor on the optimal clear day": len(DAYS["clear"]["acted"]),
    "reserve: extra cost of the clear day planned with the reserve (euro)":
        round(round(DAYS["clear"]["planned"]["cost"], 2) - round(DAYS["clear"]["best"]["cost"], 2), 2),
}
for key, value in STAGE14.items():
    print(f"{key:<70} {value}")

**Python note.** Every value of `STAGE14` is computed again from the functions of the earlier sections, so the safety case changes with them if a design or a model changes.

In [ ]:
# Figure: the safety case of the island supply with its evidence
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.set_xlim(0, 11)
ax.set_ylim(0, 4.6)
ax.axis("off")
def node(x, y, w, h, text, kind):
    style = {"goal": ("#E6F0F8", "#1E6FA8"), "strategy": ("#FFF4E0", "#C98510"), "evidence": ("#E7F4EC", "#2E8B57"),
             "context": ("#F1F3F5", "#56616D")}[kind]
    shape = plt.Rectangle((x - w / 2, y - h / 2), w, h, fc=style[0], ec=style[1], lw=1.4)
    if kind == "evidence":
        shape = plt.Circle((x, y), h / 2 + 0.05, fc=style[0], ec=style[1], lw=1.4)
    ax.add_patch(shape)
    ax.text(x, y, text, ha="center", va="center", fontsize=7.4, wrap=True)
def link(a, b):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="-|>", lw=0.8, color="#1B2430"))
s = STAGE14
node(5.5, 4.15, 4.6, 0.55, "G1: The critical loads are supplied safely\nduring outages of the grid", "goal")
node(9.6, 4.15, 2.4, 0.55, "C1: 2 outages a year, events independent\nexcept the common cause", "context")
node(5.5, 3.25, 4.2, 0.5, "S1: Argue over the two hazards and the reserve", "strategy")
node(1.9, 2.25, 3.2, 0.6, "G2: The battery never energises\nthe lines of the grid", "goal")
node(5.5, 2.25, 3.2, 0.6, f"G3: The supply is lost less than\n{TARGET} times a year", "goal")
node(9.1, 2.25, 3.2, 0.6, "G4: The battery keeps\nits reserve of 130 kWh", "goal")
node(1.9, 0.75, 1.6, 1.0, f"Model checking:\n{s['transfer logic, corrected: reachable states']} states,\n"
     f"{s['transfer logic, corrected: unsafe states']} unsafe", "evidence")
node(4.6, 0.75, 1.6, 1.0, f"Fault tree:\n{s['critical supply, chosen design: losses per year']:.4f}\nper year", "evidence")
node(6.4, 0.75, 1.6, 1.0, f"50000 years:\n{s['critical supply, chosen design: simulated losses per year']:.4f}\nper year",
     "evidence")
node(9.1, 0.75, 1.6, 1.0, f"Monitor and\ntests: {s['reserve: interventions of the monitor on the optimal clear day']}"
     "\ninterventions", "evidence")
for a, b in (((5.5, 3.87), (5.5, 3.5)), ((7.8, 4.15), (8.4, 4.15)), ((5.5, 3.0), (1.9, 2.55)), ((5.5, 3.0), (5.5, 2.55)),
             ((5.5, 3.0), (9.1, 2.55)), ((1.9, 1.95), (1.9, 1.3)), ((5.5, 1.95), (4.6, 1.3)), ((5.5, 1.95), (6.4, 1.3)),
             ((9.1, 1.95), (9.1, 1.3))):
    link(a, b)
plt.tight_layout()
plt.show()

**Exercise E.** Store the mean number of years between two losses of the critical supply with the chosen design, computed from `STAGE14`, in `y_e`.

In [ ]:
y_e = None   # your answer

check("Exercise E", y_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
y_e = 1 / STAGE14['critical supply, chosen design: losses per year']

check("Exercise E", y_e)

**Observe and reflect.** Which piece of evidence of the safety case is the weakest, and what would you do to make it stronger?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE14" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-14/"
               "NB14_dependability_safety_verification.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Every system that must keep working when parts fail has a fault tree. Its minimal cut sets, the smallest combinations of basic events that cause the top event, show where it is weak. The function of this section computes the probability of the top event and the importance of each event for a system of your choice.

The systems, their events and their probabilities are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "hospital_backup"
# choose: hospital_backup, data_center_cooling, elevator_brake, railway_crossing, pumping_station, traffic_lights
APPS = {  # top event, probabilities of the basic events on demand, minimal cut sets
    "hospital_backup": ("a hospital ward loses its power in an outage of the grid",
                        {"transfer switch fails": 0.004, "generator fails to start": 0.03, "fuel contaminated": 0.003,
                         "UPS battery empty": 0.02, "ward panel fails": 0.001},
                        [("transfer switch fails",), ("generator fails to start", "UPS battery empty"),
                         ("fuel contaminated", "UPS battery empty"), ("ward panel fails",)]),
    "data_center_cooling": ("a data center loses its cooling",
                            {"chiller 1 fails": 0.05, "chiller 2 fails": 0.05, "common cause of the chillers": 0.005,
                             "pump fails": 0.01, "controller fails": 0.002},
                            [("chiller 1 fails", "chiller 2 fails"), ("common cause of the chillers",), ("pump fails",),
                             ("controller fails",)]),
    "elevator_brake": ("an elevator brake fails to hold",
                       {"brake circuit 1 fails": 0.01, "brake circuit 2 fails": 0.01, "common cause of the brakes": 0.0005,
                        "controller fails": 0.001, "safety relay fails": 0.002},
                       [("brake circuit 1 fails", "brake circuit 2 fails"), ("common cause of the brakes",),
                        ("controller fails", "safety relay fails")]),
    "railway_crossing": ("the barriers of a crossing stay open when a train comes",
                         {"sensor A fails": 0.02, "sensor B fails": 0.02, "motor fails": 0.003, "controller fails": 0.001,
                          "mains fails": 0.01, "battery fails": 0.05},
                         [("sensor A fails", "sensor B fails"), ("motor fails",), ("controller fails",),
                          ("mains fails", "battery fails")]),
    "pumping_station": ("a town loses its water pressure",
                        {"pump 1 fails": 0.04, "pump 2 fails": 0.04, "level sensor fails": 0.005, "mains fails": 0.01,
                         "diesel fails": 0.05, "valve stuck": 0.002},
                        [("pump 1 fails", "pump 2 fails"), ("level sensor fails",), ("mains fails", "diesel fails"),
                         ("valve stuck",)]),
    "traffic_lights": ("two directions get green at the same time",
                       {"controller fault": 0.01, "conflict monitor fails": 0.001, "wiring fault": 0.002},
                       [("controller fault", "conflict monitor fails"), ("wiring fault", "conflict monitor fails")]),
}

def app_tree(app):
    top, q, cuts = APPS[app]
    p_cut = [float(np.prod([q[e] for e in c])) for c in cuts]
    rare = sum(p_cut)                                       # the rare event approximation
    bound = 1 - float(np.prod([1 - p for p in p_cut]))      # the upper bound of the minimal cut sets
    print(f"{app}: top event: {top}")
    print(f"  P(top) = {rare:.5f} (rare event approximation), {bound:.5f} (upper bound)")
    for c, p in sorted(zip(cuts, p_cut), key=lambda x: -x[1]):
        print(f"  cut set {' AND '.join(c):<58} {p:.6f}")
    for e in q:
        share = sum(p for c, p in zip(cuts, p_cut) if e in c) / rare
        print(f"  importance of {e:<32} {share:.3f}")

app_tree(APP)

**Python note.** The function `sorted` with `key=lambda x: -x[1]` orders the cut sets by their probability, the largest first. The expression `' AND '.join(c)` writes the events of a cut set with the word AND between them.

**Observe and reflect.** Which single change would lower the probability of the top event most in your system, and how do the importances show it?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write the transfer logic of Section 1 as a table of transitions instead of if statements, and let the model checker read the table. |
| Embedded systems | The island controller runs a scan cycle of 10 ms. Model the timer of 300 ms in steps of 10 ms and compare the number of reachable states. |
| Internet of Things | The breaker reports its position in a network message that can be lost. Add the lost message to the model and check whether the corrected logic stays safe. |
| Computer networks | The two island controllers exchange heartbeats. Model a lost heartbeat and look for a sequence in which each controller thinks that the other one is in charge. |
| Control engineering | Add a liveness check: After the grid returns, the controller must reach GRID within five steps. Find the states that break it. |
| Data science | Estimate the failure rate of the controllers from a log of proof tests by maximum likelihood and carry its uncertainty into the fault tree. |
| Cybersecurity | An attacker can forge the reported position of the breaker once per outage. Add the attack to the model and check both versions of the logic. |
| Artificial intelligence | Train a small policy that discharges the battery for profit, put the monitor of Section 4 in front of it as a shield and count its interventions over a month. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-14/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-14).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>